In [0]:
%pip install xgboost

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python 

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, when, length, lpad, concat, year, month, sum, trim
 
# Initialize Spark session
spark = SparkSession.builder.appName("Meses").getOrCreate()
 
# Set configuration
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")


# Define paths

paths1 = [
    "/mnt/dwhblobstrg/01_INPUT/HANA/BW/HECHOS/AZDSOAVEPE2/2024/"
]

paths2 = [
    "/mnt/dwhblobstrg/01_INPUT/HANA/BW/HECHOS/AZDSOAVEPE2/2025/"
]


#Loading

df_data1 = spark.read.format("parquet").option("recursiveFileLookup", "true").load(paths1)
df_data2 = spark.read.format("parquet").option("recursiveFileLookup", "true").load(paths2)


# Ensure all DataFrames have the same columns
common_columns = list(set(df_data1.columns) & set(df_data2.columns))

df_data1 = df_data1.select(common_columns)
df_data2 = df_data2.select(common_columns)

# Union
df_data = df_data1.union(df_data2)


# Cell 1: Filter and select relevant columns
df_datafilter = df_data.select(
    col("/BIC/ZCHASPLAN"),
    col("PLANT"),
    col("/BIC/ZCHCLIEAS"),
    col("CUSTOMER"),
    col("APO_RELDAT"),
    col("/BIC/ZCHMOREPO"),
    col("/BIC/ZCHMORENT"),
    col("/BIC/ZCHCATAS4"),
    col("/BIC/ZCHPRECON"),
    col("/BIC/ZKFCANVEN"),
    col("/BIC/ZKFCAVUN"),
    col("/BIC/ZKFCANVEU"),
    col("/BIC/ZKFVRVEB"),
    col("/BIC/ZKFCANLIT"),
    col("/BIC/ZKFFACHEC"),
    col("/BIC/ZCHASOPER"),
    col("/BIC/ZCHSABAS4"),
    col("/BIC/ZCHMAAS"),
    col("MATERIAL"),
)


df_datafilter = df_datafilter.filter(col("/BIC/ZCHMORENT") == " ")


# Cell 3: Rename columns using withColumnRenamed
df_datafilter = df_datafilter \
    .withColumnRenamed("/BIC/ZCHASPLAN", "COD_PLANTA") \
    .withColumnRenamed("PLANT", "COD_PLANTA_SAP") \
    .withColumnRenamed("/BIC/ZCHCLIEAS", "CD_CLIENTE") \
    .withColumnRenamed("CUSTOMER", "CD_CLIENTE_SAP") \
    .withColumnRenamed("APO_RELDAT", "Fecha_Entrega") \
    .withColumnRenamed("/BIC/ZCHMOREPO", "CD_MOTIVO_REP") \
    .withColumnRenamed("/BIC/ZCHMORENT", "Motivo_Rechazo") \
    .withColumnRenamed("/BIC/ZCHCATAS4", "Categoria") \
    .withColumnRenamed("/BIC/ZCHPRECON", "CD_Presentacion1") \
    .withColumnRenamed("/BIC/ZKFCANVEN", "Cajas_Fisicas_Venta") \
    .withColumnRenamed("/BIC/ZKFCAVUN", "Transacciones_Venta") \
    .withColumnRenamed("/BIC/ZKFCANVEU", "Cajas_Unitarias_Venta") \
    .withColumnRenamed("/BIC/ZKFVRVEB", "Valor_Neto_Venta") \
    .withColumnRenamed("/BIC/ZKFCANLIT", "Litros_Venta") \
    .withColumnRenamed("/BIC/ZKFFACHEC", "Factor_de_Conversion_HL") \
    .withColumnRenamed("/BIC/ZCHASOPER", "CD_OPERACION") \
    .withColumnRenamed("/BIC/ZCHSABAS4", "CD_SABOR1") \
    .withColumnRenamed("/BIC/ZCHMAAS", "CD_PRODUCTO1") \
    .withColumnRenamed("MATERIAL", "CD_PRODUCTO2")


df_datafilter = df_datafilter.withColumn(
    "Valor neto sin rep",
    when(
        (col("CD_OPERACION") == "01") |
        (col("CD_OPERACION") == "89") |
        (col("CD_OPERACION") == "02"),
        col("Valor_Neto_Venta")
    ).otherwise(0)
)
   
df_datafilter = df_datafilter.withColumn(
    "Cajas unitarias sin Rep",
    when(
        ((col("CD_OPERACION") == "89") |
        (col("CD_OPERACION") == "01")) &
        (col("CD_MOTIVO_REP") == " ") |
        ((col("CD_MOTIVO_REP") == "Z03") |
        (col("CD_MOTIVO_REP") == "Z04") |
        (col("CD_MOTIVO_REP") == "Z18") |
        (col("CD_MOTIVO_REP") == "Z19")),
        col("Cajas_Unitarias_Venta")
    ).otherwise(0)
)
 
df_datafilter = df_datafilter.withColumn(
    "Cajas fisicas sin Rep",
    when((
        (col("CD_OPERACION") == "89") |
        (col("CD_OPERACION") == "01")) &
        (col("CD_MOTIVO_REP") == " ") |
        ((col("CD_MOTIVO_REP") == "Z03") |
        (col("CD_MOTIVO_REP") == "Z04") |
        (col("CD_MOTIVO_REP") == "Z18") |
        (col("CD_MOTIVO_REP") == "Z19")),  
        col("Cajas_Fisicas_Venta")
    ).otherwise(0)
)
 
df_datafilter = df_datafilter.withColumn(
    "Transacciones sin Rep",
    when(
        (
        (col("CD_OPERACION") == "89") |
        (col("CD_OPERACION") == "01")) &
        (col("CD_MOTIVO_REP") == " ") |
        ((col("CD_MOTIVO_REP") == "Z03") |
        (col("CD_MOTIVO_REP") == "Z04") |
        (col("CD_MOTIVO_REP") == "Z18") |
        (col("CD_MOTIVO_REP") == "Z19")),
        col("Transacciones_Venta")
    ).otherwise(0)
)
 
df_datafilter = df_datafilter.withColumn(
    "Litros sin Rep",
    when(
        (
        (col("CD_OPERACION") == "89") |
        (col("CD_OPERACION") == "01")) &
        (col("CD_MOTIVO_REP") == " ") |
        ((col("CD_MOTIVO_REP") == "Z03") |
        (col("CD_MOTIVO_REP") == "Z04") |
        (col("CD_MOTIVO_REP") == "Z18") |
        (col("CD_MOTIVO_REP") == "Z19")),
        col("Litros_Venta")
    ).otherwise(0)
)


df_datafilter = df_datafilter.withColumn(
    "CU Colocadas",
    when(
        (col("CD_OPERACION") == "01") |
        (col("CD_OPERACION") == "89") |
        (col("CD_OPERACION") == "02"),
        col("Cajas_Unitarias_Venta")
    ).otherwise(0)
)



# Define input paths for dimensions
INPUT_PATH_presentacion = "/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/MAESTRO_PRESENTACION.parquet"
INPUT_PATH_geo = "/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/MAESTRO_GEOGRAFIA.parquet"
INPUT_PATH_clientes = "/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/MAESTRA_CLIENTES.parquet"
INPUT_PATH_sabor = "/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/MAESTRO_SABOR.parquet"
INPUT_PATH_material = "/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/MAESTRO_MATERIALES.parquet"
 
# Load dimension data
df_data_presentacion = spark.read.format("parquet").option("recursiveFileLookup", "true").load(INPUT_PATH_presentacion)
df_data_geo = spark.read.format("parquet").option("recursiveFileLookup", "true").load(INPUT_PATH_geo)
df_data_clientes = spark.read.format("parquet").option("recursiveFileLookup", "true").load(INPUT_PATH_clientes)
df_data_sabor = spark.read.format("parquet").option("recursiveFileLookup", "true").load(INPUT_PATH_sabor)
df_data_material = spark.read.format("parquet").option("recursiveFileLookup", "true").load(INPUT_PATH_material)



df_data_sabor = df_data_sabor.withColumnRenamed("CD_SABOR", "CD_SABOR_MAESTRO")
df_data_presentacion = df_data_presentacion.withColumnRenamed("CD_PRESENTACION", "CD_PRESENTACION_MAESTRO")
df_data_material = df_data_material.select("CD_PROD_SAP", "CD_PROD_400", "CD_PRODUCTO", "DS_NOMBRE", "CD_PRESENTACION", "CD_SABOR", "DS_UEN", "DS_SUBINDUSTRIA", "DS_CIA")

df_dimensiones_totales = df_data_material.join(df_data_presentacion, df_data_material["CD_PRESENTACION"] == df_data_presentacion["CD_PRESENTACION_MAESTRO"], "left")

df_dimensiones_totales = df_dimensiones_totales.join(df_data_sabor, df_dimensiones_totales["CD_SABOR"] == df_data_sabor["CD_SABOR_MAESTRO"],"left")



#Pegando todo a la transaccional

df_final = df_datafilter.join(df_dimensiones_totales, df_datafilter["CD_PRODUCTO1"] == df_dimensiones_totales["CD_PROD_400"], "left")

df_data_geo = df_data_geo.select("CD_PLANTA", "DS_PLANTA","DS_TERRITORIO")

df_final = df_final.join(df_data_geo, df_final["COD_PLANTA"] == df_data_geo["CD_PLANTA"], "left")

df_final = df_final.filter((col("DS_TERRITORIO") == "Occidente") | (col("DS_TERRITORIO") == "Antioquia") | (col("DS_TERRITORIO") == "Costa") | (col("DS_TERRITORIO") == "Centro Norte") | (col("DS_TERRITORIO") == "Centro Sur") | (col("DS_TERRITORIO") == "Santander"))


#Filtrando sólo los campos que necesitamos

df_final = df_final.select("COD_PLANTA", "COD_PLANTA_SAP", "CD_CLIENTE", "CD_CLIENTE_SAP","Fecha_Entrega", "Cajas_Fisicas_Venta", "Transacciones_Venta", "Cajas_Unitarias_Venta", "Valor_Neto_Venta", "Litros_Venta", "Factor_de_Conversion_HL", "Valor neto sin rep", "Cajas unitarias sin Rep", "Cajas fisicas sin Rep", "Transacciones sin Rep", "Litros sin Rep", "CU Colocadas", "DS_PRESENTACION", "DS_CATEGORIA", "DS_SABOR", "DS_MARCA", "DS_PLANTA", "DS_TERRITORIO", "DS_SUBINDUSTRIA", "DS_CIA", "DS_CLASE", "DS_SUBTENVASE", "DS_SUBCATEGORIA", "DS_GPRESENTACION_MC1", "DS_CLASIFICACION_BEBIDA", "DS_MAXI_MINI", "CD_PRODUCTO1", "CD_PRODUCTO2")


#CREANDO LA LLAVE DEL CLIENTE (2 caminos: AS400 y DSD)

df_final = df_final.withColumn("TIPO_CEDI", 
                                when((col("COD_PLANTA") == "0000") | (col("COD_PLANTA") == "1836") | (col("COD_PLANTA") == "0116") | (col("COD_PLANTA") == "0124") | (col("COD_PLANTA") == "1030") | (col("COD_PLANTA") == "1535") | (col("COD_PLANTA") == "1029") | (col("COD_PLANTA") == "1027") | (col("COD_PLANTA") == "1028") | (col("COD_PLANTA") == "1534") | (col("COD_PLANTA") == "1533") | (col("COD_PLANTA") == "0121") | (col("COD_PLANTA") == "0125") | (col("COD_PLANTA") == "0103") | (col("COD_PLANTA") == "0108") | (col("COD_PLANTA") == "0120") | (col("COD_PLANTA") == "0126") | (col("COD_PLANTA") == "0127") | (col("COD_PLANTA") == "0101") | (col("COD_PLANTA") == "2240") | (col("COD_PLANTA") == "1332") | (col("COD_PLANTA") == "1331") | (col("COD_PLANTA") == "1330") | (col("COD_PLANTA") == "0113") | (col("COD_PLANTA") == "0104") | (col("COD_PLANTA") == "0115") | (col("COD_PLANTA") == "0107") | (col("COD_PLANTA") == "0430") | (col("COD_PLANTA") == "2038") | (col("COD_PLANTA") == "0826") | (col("COD_PLANTA") == "0827") | (col("COD_PLANTA") == "0227") | (col("COD_PLANTA") == "0426") | (col("COD_PLANTA") == "0422") | (col("COD_PLANTA") == "0428") | (col("COD_PLANTA") == "0429") | (col("COD_PLANTA") == "0421") | (col("COD_PLANTA") == "B358") | (col("COD_PLANTA") == "B292") | (col("COD_PLANTA") == "B460") | (col("COD_PLANTA") == "0112") | (col("COD_PLANTA") == "0122") | (col("COD_PLANTA") == "0927") | (col("COD_PLANTA") == "0825") | (col("COD_PLANTA") == "0123") | (col("COD_PLANTA") == "0043") | (col("COD_PLANTA") == "0106") | (col("COD_PLANTA") == "0114") | (col("COD_PLANTA") == "0215") | (col("COD_PLANTA") == "B111") | (col("COD_PLANTA") == "B422") | (col("COD_PLANTA") == "1128") | (col("COD_PLANTA") == "1129") | (col("COD_PLANTA") == "2971") | (col("COD_PLANTA") == "2972") | (col("COD_PLANTA") == "1634") | (col("COD_PLANTA") == "1637") | (col("COD_PLANTA") == "1636") | (col("COD_PLANTA") == "1638") | (col("COD_PLANTA") == "1635") | (col("COD_PLANTA") == "1229") | (col("COD_PLANTA") == "1230") | (col("COD_PLANTA") == "1735") | (col("COD_PLANTA") == "1737"), "DSD")
                                .otherwise("AS400"))


# Reemplazar ceros iniciales en la columna 'CD_CLIENTE'
df_llave_cliente = df_final.withColumn(
    "CD_CLIENTEE",
    regexp_replace(col("CD_CLIENTE"), "^0+", "")
)


# DSD

df_llave_cliente = df_llave_cliente.withColumn(
    "LLAVE_CLIENTE",
    when(col("TIPO_CEDI") == "DSD", concat(col("COD_PLANTA"), col("CD_CLIENTE_SAP")))
    .otherwise(concat(col("COD_PLANTA"), col("CD_CLIENTEE"))))

df_llave_cliente_DSD = df_llave_cliente.filter(col("TIPO_CEDI") == "DSD")



#Trayendo el DF de clientes

INPUT_PATH_clientes = "/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/MAESTRA_CLIENTES.parquet"
df_data_clientes = spark.read.format("parquet").option("recursiveFileLookup", "true").load(INPUT_PATH_clientes)

df_data_clientes = df_data_clientes.select("CD_CLIENTE_SAP", "CD_PLANTA", "CD_CLIENTE")

df_data_clientes = df_data_clientes.withColumn("LLAVE_CLIENTE_SAP", concat(col("CD_PLANTA"), col("CD_CLIENTE_SAP")))
df_data_clientes = df_data_clientes.withColumn("LLAVE_CLIENTE_AS400", concat(col("CD_PLANTA"), col("CD_CLIENTE")))

df_data_clientes = df_data_clientes.withColumnRenamed("CD_CLIENTE", "CD_CLIENTE_400")

df_data_clientes = df_data_clientes.select("LLAVE_CLIENTE_SAP","LLAVE_CLIENTE_AS400","CD_CLIENTE_400")


#Pegando los DFs DSD

df_llave_cliente_DSD = df_llave_cliente_DSD.join(df_data_clientes, df_llave_cliente_DSD["LLAVE_CLIENTE"] == df_data_clientes["LLAVE_CLIENTE_SAP"], "left")

df_llave_cliente_DSD = df_llave_cliente_DSD.drop("LLAVE_CLIENTE_SAP")






# AS400

df_llave_cliente_AS400 = df_llave_cliente.filter(col("TIPO_CEDI") == "AS400")

# Add a new column with the length of the 'CD_CLIENTE' field
df_llave_cliente_AS400 = df_llave_cliente_AS400.withColumn("CD_CLIENTE_length", length(col("CD_CLIENTEE")))

df_llave_cliente_AS400 = df_llave_cliente_AS400.drop("LLAVE_CLIENTE")


# Create the new 'CD_CLIENTEE' field with the specified conditions
df_llave_cliente_AS400 = df_llave_cliente_AS400.withColumn(
    "CD_CLIENTEE2",
    when(col("CD_CLIENTE_length") == 1, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 2, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 3, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 4, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 5, lpad(col("CD_CLIENTEE"), 6, '0'))
    .otherwise(col("CD_CLIENTEE"))
)


df_llave_cliente_AS400 = df_llave_cliente_AS400.withColumn("LLAVE_CLIENTE", concat(col("COD_PLANTA"), col("CD_CLIENTEE2")))
df_llave_cliente_AS400 = df_llave_cliente_AS400.withColumn("LLAVE_CLIENTE_AS400", col("LLAVE_CLIENTE"))
df_llave_cliente_AS400 = df_llave_cliente_AS400.withColumn("CD_CLIENTE_400", col("CD_CLIENTEE2"))

df_llave_cliente_AS400 = df_llave_cliente_AS400.drop("CD_CLIENTE_length", "CD_CLIENTEE2")


#Pegando Dfs DSD y AS400

df_final_Cliente = df_llave_cliente_DSD.union(df_llave_cliente_AS400)


df_final_Cliente = df_final_Cliente.withColumn(
    "LLAVE_CLIENTE_3_SAP",
    when(col("TIPO_CEDI") == "DSD", concat(col("COD_PLANTA_SAP"), col("CD_CLIENTE_SAP")))
    .otherwise((col("LLAVE_CLIENTE"))))


# Convert 'Fecha Entrega' to datetime and extract year and month
df_final_Cliente = df_final_Cliente.withColumn("Fecha Entrega", col("Fecha_Entrega").cast("date"))
df_final_Cliente = df_final_Cliente.withColumn("Anio", year(col("Fecha Entrega")).cast("string"))
df_final_Cliente = df_final_Cliente.withColumn("Mes", lpad(month(col("Fecha Entrega")).cast("string"), 2, '0'))
df_final_Cliente = df_final_Cliente.withColumn("Anio_Mes", concat(col("Anio"), col("Mes")))


#PEGANDO MACROSEGMENTOS

#CREAMOS Segmento Pos (remplazo de segmento CIA)
path_dim_sgto_pos = '/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/BIC_PZCHCLIEAS'
path_mto_macroseg = '/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/BIC_TZCHMACROS'
path_mto_mision_compra = '/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/BIC_TZCHMISCO'
path_mto_subtipologia = '/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/BIC_TZCHSBTIP'
path_mto_tipologia = '/mnt/dwhblobstrg/01_INPUT/HANA/BW/DIMENSIONES/BIC_TZCHTISEG'

# Load dimension data
df_dim_sgto_pos = spark.read.format("parquet").option("recursiveFileLookup", "true").load(path_dim_sgto_pos)
df_mto_macroseg = spark.read.format("parquet").option("recursiveFileLookup", "true").load(path_mto_macroseg)
df_mto_mision_compra = spark.read.format("parquet").option("recursiveFileLookup", "true").load(path_mto_mision_compra)
df_mto_subtipologia = spark.read.format("parquet").option("recursiveFileLookup", "true").load(path_mto_subtipologia)
df_mto_tipologia = spark.read.format("parquet").option("recursiveFileLookup", "true").load(path_mto_tipologia)


nuevos_nombres = ['CD_PLANTA1', 'CD_CLIENTE1', 'OBJVERS', 'CHANGED', 'IND_PREVENTA', 'CD_CLIENTE_SAP1', 'BL_RETIRO', 'ABC', 'NRONIT','NRO_TEL_FIJO','DIRECCIOM', 'BL_LONGITUD','BL_LATITUD','NRO_CELULAR','EMAIL','CD_TIPO_NEGOCIO','CD_DEPARTAMENTO','CD_MUNICIPIO','DS_NOMBRE_CLIENTE','DS_NOMBRE_NEGOCIO','CD_SEGMENTO','CD_CLASE','VISITAS_PROG_ALFA','VISITAS_PROG_DELTA','CLIRIME','CD_MOD_AT','CLUSTER_SEGM_SHOPPER','MARCA_PIL_SHOPPER','ESTRATO','CD_MACROSEGMENTO1','CD_MISION_COMPRA','CD_TIPOLOGIA1','CD_SUBTIPOLOGIA1']

# Obtener los nombres actuales de las columnas
nombres_actuales = df_dim_sgto_pos.columns

# Asegurarse de que la longitud de la lista de nuevos nombres coincida con la longitud de los nombres actuales
if len(nuevos_nombres) != len(nombres_actuales):
    raise ValueError("La lista de nuevos nombres debe tener la misma longitud que la lista de nombres actuales")

# Renombrar las columnas
for nombre_actual, nuevo_nombre in zip(nombres_actuales, nuevos_nombres):
    df_dim_sgto_pos = df_dim_sgto_pos.withColumnRenamed(nombre_actual, nuevo_nombre)


df_mto_macroseg = df_mto_macroseg.withColumnRenamed("/BIC/ZCHMACROS", "CD_MACROSEGMENTO")
df_mto_macroseg = df_mto_macroseg.withColumnRenamed("TXTLG", "MACROSEGMENTO")

df_mto_mision_compra = df_mto_mision_compra.withColumnRenamed("/BIC/ZCHMISCO", "CD_MISION_COMPRA2")
df_mto_mision_compra = df_mto_mision_compra.withColumnRenamed("TXTLG", "MISION_COMPRA")

df_mto_subtipologia = df_mto_subtipologia.withColumnRenamed("/BIC/ZCHSBTIP", "CD_SUBTIPOLOGIA")
df_mto_subtipologia = df_mto_subtipologia.withColumnRenamed("TXTLG", "SUBTIPOLOGIA")

df_mto_tipologia = df_mto_tipologia.withColumnRenamed("/BIC/ZCHTISEG", "CD_TIPOLOGIA")
df_mto_tipologia = df_mto_tipologia.withColumnRenamed("TXTLG", "TIPOLOGIA")


df_macroseg = df_mto_macroseg.filter(col("CD_MACROSEGMENTO") != ' ')
df_mision_compra = df_mto_mision_compra.filter(col("CD_MISION_COMPRA2") != ' ')
df_subtipologia = df_mto_subtipologia.filter(col("CD_SUBTIPOLOGIA") != ' ')
df_tipologia = df_mto_tipologia.filter(col("CD_TIPOLOGIA") != ' ')

#Agregamos las descripciones a Segmentos POS
df_sgto_pos = df_dim_sgto_pos.select('CD_PLANTA1', 'CD_CLIENTE1', 'CD_CLIENTE_SAP1', 'CD_MACROSEGMENTO1', 'CD_MISION_COMPRA', 'CD_TIPOLOGIA1', 'CD_SUBTIPOLOGIA1', 'ABC')

#Join con Macrosegmentos
df_sgto_pos = df_sgto_pos.join(df_macroseg, df_sgto_pos["CD_MACROSEGMENTO1"] == df_macroseg["CD_MACROSEGMENTO"], "left")

#Join con Tipologia
df_sgto_pos = df_sgto_pos.join(df_tipologia, df_sgto_pos["CD_TIPOLOGIA1"] == df_tipologia["CD_TIPOLOGIA"], "left")

#Join con Subtipologia
df_sgto_pos = df_sgto_pos.join(df_subtipologia, df_sgto_pos["CD_SUBTIPOLOGIA1"] == df_subtipologia["CD_SUBTIPOLOGIA"], "left")

df_sgto_pos = df_sgto_pos.select('CD_PLANTA1', 'CD_CLIENTE1', 'CD_CLIENTE_SAP1', 'MACROSEGMENTO','TIPOLOGIA', 'SUBTIPOLOGIA')


#CREANDO LA LLAVE DEL CLIENTE (MACROSEGMENTOS)

# Reemplazar ceros iniciales en la columna 'CD_CLIENTE'
df_llave_cliente = df_sgto_pos.withColumn(
    "CD_CLIENTEE",
    regexp_replace(col("CD_CLIENTE1"), "^0+", "")
)

# Add a new column with the length of the 'CD_CLIENTE' field
df_llave_cliente = df_llave_cliente.withColumn("CD_CLIENTE_length", length(col("CD_CLIENTEE")))

# Create the new 'CD_CLIENTEE' field with the specified conditions
df_llave_cliente = df_llave_cliente.withColumn(
    "CD_CLIENTEE2",
    when(col("CD_CLIENTE_length") == 1, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 2, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 3, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 4, lpad(col("CD_CLIENTEE"), 6, '0'))
    .when(col("CD_CLIENTE_length") == 5, lpad(col("CD_CLIENTEE"), 6, '0'))
    .otherwise(col("CD_CLIENTEE"))
)


df_llave_cliente = df_llave_cliente.withColumn("LLAVE_CLIENTE_AS400_2", concat(col("CD_PLANTA1"), col("CD_CLIENTEE2")))

df_llave_cliente = df_llave_cliente.select('LLAVE_CLIENTE_AS400_2','MACROSEGMENTO','TIPOLOGIA', 'SUBTIPOLOGIA')




#Pegando MACRO CON TRX

df_final_Cliente = df_final_Cliente.join(df_llave_cliente, df_final_Cliente["LLAVE_CLIENTE_AS400"] == df_llave_cliente["LLAVE_CLIENTE_AS400_2"], "left")

df_final_Cliente = df_final_Cliente.drop("LLAVE_CLIENTE_AS400_2")



df_final_Cliente = df_final_Cliente.withColumn("AÑOMES", col("Anio_Mes").cast("integer"))

In [0]:
df_final_Cliente = df_final_Cliente.filter(
    (df_final_Cliente.AÑOMES == 202501) |
    (df_final_Cliente.AÑOMES == 202502) |
    (df_final_Cliente.AÑOMES == 202503) |
    (df_final_Cliente.AÑOMES == 202504) |
    (df_final_Cliente.AÑOMES == 202505) |
    (df_final_Cliente.AÑOMES == 202506) |
    (df_final_Cliente.AÑOMES == 202507) |
    (df_final_Cliente.AÑOMES == 202508) |
    (df_final_Cliente.AÑOMES == 202509) |
    (df_final_Cliente.AÑOMES == 202510) |
    (df_final_Cliente.AÑOMES == 202511) |
    (df_final_Cliente.AÑOMES == 202512)
)

In [0]:
%python
from pyspark.sql.functions import countDistinct, sum, count

aagg = df_final_Cliente.groupBy("LLAVE_CLIENTE", "LLAVE_CLIENTE_AS400", "LLAVE_CLIENTE_3_SAP", "CD_PRODUCTO1", "Fecha_Entrega", "TIPO_CEDI").agg(
    count("CD_PRODUCTO1").alias("quantity")
)

Basefinal = aagg.withColumn(
    "LLAVE",
    when(col("TIPO_CEDI") == "AS400", col("LLAVE_CLIENTE_AS400"))
    .otherwise(col("LLAVE_CLIENTE_3_SAP"))
)

Basefinal = Basefinal.select('LLAVE', 'CD_PRODUCTO1', 'Fecha_Entrega', 'quantity')

Basefinal = Basefinal.withColumnRenamed("LLAVE", "client_id")
Basefinal = Basefinal.withColumnRenamed("CD_PRODUCTO1", "product_id")
Basefinal = Basefinal.withColumnRenamed("Fecha_Entrega", "order_date")

In [0]:
display(Basefinal)

client_id,product_id,order_date,quantity
00431002240149,0404,2025-06-29,1
00431002240149,3894,2025-07-23,1
00431002240149,0311,2025-08-06,1
00431002240149,3773,2025-08-06,1
00431002240149,3891,2025-08-06,1
00431002240149,2954,2025-08-09,1
00431002240149,0235,2025-07-26,1
00431002240149,0539,2025-07-26,1
00431002240149,3887,2025-07-26,1
00431002240149,3943,2025-07-26,1


In [0]:
sales_df = Basefinal

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.ml.recommendation import ALS
import xgboost as xgb
import numpy as np
from pyspark.sql.functions import countDistinct, sum, count
from pyspark.ml.feature import StringIndexer

In [0]:
client_indexer = StringIndexer(
    inputCol="client_id",
    outputCol="client_id_idx",
    handleInvalid="skip"
)

product_indexer = StringIndexer(
    inputCol="product_id",
    outputCol="product_id_idx",
    handleInvalid="skip"
)

client_indexer_model = client_indexer.fit(sales_df)
product_indexer_model = product_indexer.fit(sales_df)

sales_indexed = client_indexer_model.transform(sales_df)
sales_indexed = product_indexer_model.transform(sales_indexed)

🏃 View run defiant-cub-719 at: https://adb-850397503872101.1.azuredatabricks.net/ml/experiments/4382422506939759/runs/a9ea4c3cebe14e6ba8b3b560e7c9b994
🧪 View experiment at: https://adb-850397503872101.1.azuredatabricks.net/ml/experiments/4382422506939759
🏃 View run wise-ray-174 at: https://adb-850397503872101.1.azuredatabricks.net/ml/experiments/4382422506939759/runs/d29f9d955c104c0d8e8dda5f2257da37
🧪 View experiment at: https://adb-850397503872101.1.azuredatabricks.net/ml/experiments/4382422506939759


In [0]:
from pyspark.sql.functions import add_months, current_date

# 1. Filtrar histórico
sales_filtered = sales_indexed.filter(
    F.col("order_date") >= add_months(current_date(), -12)
)


# Clientes activos
active_clients = (
    sales_filtered
    .groupBy("client_id_idx")
    .count()
    .filter(F.col("count") >= 3)
    .select("client_id_idx")
)

# Productos activos
active_products = (
    sales_filtered
    .groupBy("product_id_idx")
    .count()
    .filter(F.col("count") >= 5)
    .select("product_id_idx")
)

als_input_reduced = (
    als_input
    .join(active_clients, "client_id_idx")
    .join(active_products, "product_id_idx")
    .repartition(200)
)

spark.sparkContext.setCheckpointDir("/tmp/als_checkpoint")
als_input_reduced = als_input_reduced.checkpoint()

als = ALS(
    userCol="client_id_idx",
    itemCol="product_id_idx",
    ratingCol="rating",
    implicitPrefs=True,
    rank=5,
    maxIter=3,
    regParam=0.1,
    alpha=5,
    coldStartStrategy="drop"
)

als_model = als.fit(als_input_reduced)

🏃 View run upbeat-mare-928 at: https://adb-850397503872101.1.azuredatabricks.net/ml/experiments/4382422506939759/runs/56d5a6b4e8f0466696888529f1ecfb02
🧪 View experiment at: https://adb-850397503872101.1.azuredatabricks.net/ml/experiments/4382422506939759


In [0]:
candidates_idx = (
    als_model
    .recommendForAllUsers(100)
    .selectExpr("client_id_idx", "explode(recommendations) as rec")
    .select(
        "client_id_idx",
        F.col("rec.product_id_idx").alias("product_id_idx"),
        F.col("rec.rating").alias("als_score")
    )
)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
client_lookup = sales_indexed.select("client_id", "client_id_idx").distinct()
product_lookup = sales_indexed.select("product_id", "product_id_idx").distinct()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr

In [0]:
candidates = (
    candidates_idx
    .join(client_lookup, "client_id_idx", "left")
    .join(product_lookup, "product_id_idx", "left")
    .select("client_id", "product_id", "als_score")
)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:434)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:477)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:750)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:510)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:616)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:643)
	at com.databricks.logging.AttributionContextTracing.$anonfun$withAttributionContext$1(AttributionContextTracing.scala:49)
	at com.databricks.logging.AttributionContext$.$anonfun$withValue$1(AttributionContext.scala:293)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(Attr